# Install Library

In [ ]:
#!pip install pandas
#!pip install matplotlib
#!pip install seaborn
#!pip install scikit-learn
#!pip install scikit-surprise
#!pip install numpy

# Import Libray

In [ ]:
# Step 1: Import libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt
import seaborn as sns

# Load Dataset 


In [ ]:
movies = pd.read_csv(r"C:\Users\User\Desktop\movie\ml-32m\movies.csv")
ratings = pd.read_csv(r"C:\Users\User\Desktop\movie\ml-32m\ratings.csv")
tags = pd.read_csv(r"C:\Users\User\Desktop\movie\ml-32m\tags.csv")

In [ ]:
# Check the rows 

print(movies.head(10))
print(ratings.head(10))
print(tags.head(10))


In [ ]:
#Explanation:

#movies.csv → MovieID, title, genres

#ratings.csv → userId, movieId, rating, timestamp

#tags.csv → userId, movieId, tag, timestamp (optional for now)

# Missing Values

In [ ]:
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)

# Create Memory-friendly subset

In [ ]:
top_users = ratings['userId'].value_counts().head(5000).index
top_movies = ratings['userId'].value_counts().head(1000).index
ratings_small = ratings[ratings['userId'].isin(top_users) & ratings['movieId'].isin(top_movies)]

In [ ]:
# user movies matrix

user_movie_matrix = ratings_small.pivot(index='userId', columns='movieId', values='rating').fillna(0)
movie_user_matrix = user_movie_matrix.T # Transpose matrix for item based CF

In [ ]:
# Item based similarity

item_similarity = cosine_similarity(movie_user_matrix)
item_similarity_df = pd.DataFrame(item_similarity, index=movie_user_matrix.index, columns=movie_user_matrix.index)


In [ ]:
# Item based recommendation function

def recommend_movies(movie_id, top_n=5):
    similar_movies = item_similarity_df[movie_id].sort_values(ascending=False).drop(movie_id)
    top_ids = similar_movies.head(top_n).index
    return movies[movies['movieId'].isin(top_ids)][['movieId','title']]

In [ ]:
# SVD AI for personalized prediction
svd = TruncatedSVD(n_components=20, random_state=42)
svd_matrix = svd.fit_transform(user_movie_matrix)
predicted_ratings = np.dot(svd_matrix, svd.components_)
predicted_ratings_df = pd.DataFrame(predicted_ratings, index=user_movie_matrix.index, columns=user_movie_matrix.columns)


In [ ]:
# Hybrid Recommendation Function

def hybrid_recommend(user_id, top_n=5):
    # Predicted ratings for this user
    user_pred = predicted_ratings_df.loc[user_id].sort_values(ascending=False)
    
    #  Top N movie IDs for this user
    top_movies_ids = user_pred.head(top_n).index  # head() यहाँ Series मा apply हुन्छ, user_id होइन
    
    #  Create recommended DataFrame
    recommended = pd.DataFrame()
    recommended['movieId'] = top_movies_ids
    recommended['predicted_rating'] = user_pred[top_movies_ids].values
    recommended = recommended.merge(movies[['movieId', 'title']], on='movieId')
    
    return recommended

In [ ]:
# Data Visualization
# Ratings Distribution

plt.figure(figsize=(8,5))
sns.histplot(ratings['rating'], bins=10, kde=False)
plt.title("Ratings Distribution")
plt.show()


In [ ]:
# Top 10 popular movies

popular = ratings.groupby('movieId')['rating'].count().sort_values(ascending=False).head(10)
popular_movies = movies[movies['movieId'].isin(popular.index)]
plt.figure(figsize=(10,5))
sns.barplot(x=popular_movies['title'],y=popular.values)
plt.xticks(rotation=45)
plt.title("Top 10 Popular Movies")
plt.show()
           

In [ ]:
# Ratings histogram
plt.figure(figsize=(10,6))
sns.histplot(ratings['rating'], bins=10, kde=True, color='skyblue')
plt.title("Ratings Distribution")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.show()

In [ ]:
# Count of ratings per movie
popular_movies = ratings.groupby('movieId').size().sort_values(ascending=False).head(10)
top_movies_titles = movies.set_index('movieId').loc[popular_movies.index]['title']

plt.figure(figsize=(12,6))
sns.barplot(x=popular_movies.values, y=top_movies_titles.values, palette="viridis")
plt.title("Top 10 Most Rated Movies")
plt.xlabel("Number of Ratings")
plt.ylabel("Movie")
plt.show()

In [ ]:
# User input from dataset
user_id = int(input("Enter a userId from the dataset: "))

# Get top 5 recommendations for that user
user_recommendations = hybrid_recommend(user_id, top_n=5)

# Show recommendations table
print(user_recommendations)

# Plot bar chart for recommended movies
plt.figure(figsize=(10,6))
sns.barplot(x='predicted_rating', y='title', data=user_recommendations, palette='magma')
plt.title(f"Top 5 Recommendations for User {user_id}")
plt.xlabel("Predicted Rating")
plt.ylabel("Movie")
plt.show()

In [ ]:
#Test recommendation
user_test = int(input("Enter a userId from the dataset: "))
if user_test in user_movie_matrix.index:
    print(hybrid_recommend(user_test, top_n=5))
else:
    print("User ID not found. Choose from:", list(user_movie_matrix.index[:10]))